# POST-HOC / DIAGNOSTIC — Clean Own-GNN C128, Tg +115°C

Single permitted follow-up midpoint after the preregistered +40/+100/+130 probes showed a Private score minimum around +100 to +130°C. Same successful clean 25-fold ensemble; fixed offset only.

**POST-HOC / LEADERBOARD-PROBED / NOT VALID BLIND PERFORMANCE**


In [ ]:
from __future__ import annotations
import csv, hashlib, importlib.metadata, importlib.util, io, json, os, pathlib, subprocess, sys, tarfile
import numpy as np
import pandas as pd

DATASET_SLUG = "safaaa46/clean-own-gnn-c128-5seed-5fold-formal-20261004"
DATASET_DIR = pathlib.Path("/kaggle/input/datasets") / DATASET_SLUG
BUNDLE = DATASET_DIR / "asset_bundle.tar.gz"
MOUNTED_ASSET_ROOT = DATASET_DIR / "asset_bundle"
WORK = pathlib.Path("/kaggle/working")
ASSET_ROOT = WORK / "clean_own_gnn_c128_assets"
EXPECTED_FORMAL_SOURCE = "72490c1a748ed9395025f4120c9eb04f28268695"
EXPECTED_MAIN = "c9949c088d8822eac775a55ad601d278b747c9f9"
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]
DELTA = 115.0
EXPECTED_PAIRS = {(seed, fold) for seed in (42, 43, 44, 45, 46) for fold in range(5)}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

if BUNDLE.is_file():
    assert not ASSET_ROOT.exists(), f"Refusing to reuse extraction path: {ASSET_ROOT}"
    ASSET_ROOT.mkdir(parents=True)
    with tarfile.open(BUNDLE, "r:gz") as archive:
        members = archive.getmembers()
        for member in members:
            target = (ASSET_ROOT / member.name).resolve()
            assert target.is_relative_to(ASSET_ROOT.resolve()), f"Unsafe archive path: {member.name}"
        archive.extractall(ASSET_ROOT)
elif MOUNTED_ASSET_ROOT.is_dir():
    # Kaggle mounts the uploaded tar archive as an extracted asset_bundle/ tree.
    ASSET_ROOT = MOUNTED_ASSET_ROOT
else:
    # Resolve by the signed asset manifest in case Kaggle chooses a different mount name.
    input_root = pathlib.Path("/kaggle/input")
    manifest_paths = sorted(input_root.rglob("asset_manifest.json"))
    asset_roots = []
    for candidate_manifest in manifest_paths:
        candidate_root = candidate_manifest.parent
        try:
            candidate_data = json.loads(candidate_manifest.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if candidate_data.get("dataset_slug") != DATASET_SLUG:
            continue
        if all(
            (candidate_root / item["path"]).is_file()
            and (candidate_root / item["path"]).stat().st_size == item["size_bytes"]
            for item in candidate_data.get("files", [])
        ):
            asset_roots.append(candidate_root)
    if len(asset_roots) == 1:
        ASSET_ROOT = asset_roots[0]
    else:
        visible_inputs = sorted(
            str(entry.relative_to(input_root)) + ("/" if entry.is_dir() else "")
            for entry in input_root.iterdir()
        ) if input_root.is_dir() else []
        raise AssertionError(
            f"Could not uniquely locate complete private asset bundle; roots={asset_roots}; "
            f"manifests={manifest_paths}; dataset directory exists={DATASET_DIR.is_dir()}; "
            f"visible /kaggle/input={visible_inputs}"
        )

manifest_path = ASSET_ROOT / "asset_manifest.json"
manifest = json.loads(manifest_path.read_text())
assert manifest["formal_source_sha"] == EXPECTED_FORMAL_SOURCE
assert manifest["main_sha"] == EXPECTED_MAIN
assert manifest["dataset_slug"] == DATASET_SLUG
assert manifest["model_definition"]["hidden_dim"] == 128
assert manifest["model_definition"]["representation"] == "keep_dummy"
assert manifest["model_definition"]["operator"] == "GINEConv"
assert manifest["model_definition"]["num_layers"] == 4
assert manifest["model_definition"]["pooling"] == ["global_mean", "global_max"]
assert manifest["model_definition"]["run_seeds"] == [42, 43, 44, 45, 46]
assert manifest["model_definition"]["folds_per_seed"] == 5
assert len(manifest["checkpoints"]) == 25
for item in manifest["files"]:
    path = ASSET_ROOT / item["path"]
    assert path.is_file(), f"Missing payload file: {item['path']}"
    assert path.stat().st_size == item["size_bytes"]
    assert sha256(path) == item["sha256"], f"SHA256 mismatch: {item['path']}"
assert sum(p.is_file() for p in ASSET_ROOT.rglob("*.pt")) == 25
assert sum(p.is_file() for p in (ASSET_ROOT / "scalers").rglob("*.json")) == 25

# Kaggle's internet-disabled image gets both non-base dependencies from this dataset only.
wheels = ASSET_ROOT / "wheels"
def install_local_wheel(pattern):
    candidates = sorted(wheels.glob(pattern))
    assert len(candidates) == 1, f"Expected one offline wheel matching {pattern}; got {candidates}"
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--disable-pip-version-check", "--force-reinstall", str(candidates[0])], check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

rdkit_expected = "2026.03.2"
if importlib.util.find_spec("rdkit") is None:
    install_local_wheel("rdkit-2026.3.2-cp313-cp313-manylinux_2_28_x86_64.whl")
else:
    import rdkit
    if rdkit.__version__ != rdkit_expected:
        install_local_wheel("rdkit-2026.3.2-cp313-cp313-manylinux_2_28_x86_64.whl")
        del sys.modules["rdkit"]
        import importlib
        importlib.invalidate_caches()
        import rdkit
import importlib
importlib.invalidate_caches()
import rdkit
assert rdkit.__version__ == rdkit_expected, (rdkit.__version__, rdkit_expected)

try:
    pyg_version = importlib.metadata.version("torch-geometric")
except importlib.metadata.PackageNotFoundError:
    pyg_version = None
if pyg_version != "2.7.0":
    install_local_wheel("torch_geometric-2.7.0-py3-none-any.whl")
    importlib.invalidate_caches()
    pyg_version = importlib.metadata.version("torch-geometric")
assert pyg_version == "2.7.0", pyg_version

sys.path.insert(0, str(ASSET_ROOT / "source"))
import torch
import torch_geometric
from torch_geometric.loader import DataLoader
from from_scratch_gnn.models.own_gnn_repr_keep_dummy.graph import build_polymer_graph
from from_scratch_gnn.models.polymer_representation_ablation.model import OwnGNNRepresentation

assert torch.cuda.is_available(), "The requested Kaggle GPU accelerator is unavailable"
device = torch.device("cuda:0")
print("RUNTIME", json.dumps({"python": sys.version.split()[0], "torch": torch.__version__, "torch_geometric": torch_geometric.__version__, "rdkit": rdkit.__version__, "device": torch.cuda.get_device_name(device), "internet_expected": False}, sort_keys=True))

input_root = pathlib.Path("/kaggle/input")
competition_candidates = sorted({
    path.parent for path in input_root.rglob("sample_submission.csv")
    if (path.parent / "test.csv").is_file()
})
assert len(competition_candidates) == 1, (
    f"Expected one competition input directory with test.csv and sample_submission.csv; "
    f"found={competition_candidates}; visible inputs="
    f"{sorted(str(p.relative_to(input_root)) for p in input_root.iterdir())}"
)
competition_dir = competition_candidates[0]
test_path = competition_dir / "test.csv"
sample_path = competition_dir / "sample_submission.csv"
test = pd.read_csv(test_path, dtype={"id": str})
sample = pd.read_csv(sample_path, dtype={"id": str})
assert {"id", "SMILES"}.issubset(test.columns)
assert "id" in sample.columns and set(TARGETS).issubset(sample.columns)
assert len(test) == len(sample)
assert test["id"].astype(str).tolist() == sample["id"].astype(str).tolist()
assert not test["id"].isna().any() and not test["SMILES"].isna().any()
assert sample.columns.tolist() == ["id", *TARGETS]
print(f"TEST_ROWS={len(test)} SAMPLE_ROWS={len(sample)} ID_ORDER_MATCH=true COLUMNS={sample.columns.tolist()}")

print("Building test graphs using the formal-source keep-dummy graph builder...", flush=True)
graphs = []
for idx, row in enumerate(test.itertuples(index=False)):
    graph, info = build_polymer_graph(str(row.SMILES), sample_id=str(row.id))
    graphs.append(graph)
    if (idx + 1) % 1000 == 0:
        print(f"BUILT_GRAPHS={idx + 1}/{len(test)}", flush=True)
assert len(graphs) == len(test)
loader = DataLoader(graphs, batch_size=64, shuffle=False)
accumulator = np.zeros((len(test), len(TARGETS)), dtype=np.float64)
loaded_pairs = set()
loaded_hashes = []
manifest_pairs = {(int(x["run_seed"]), int(x["fold"])) for x in manifest["checkpoints"]}
assert manifest_pairs == EXPECTED_PAIRS and len(manifest_pairs) == 25

for item in sorted(manifest["checkpoints"], key=lambda x: (x["run_seed"], x["fold"])):
    run_seed, fold = int(item["run_seed"]), int(item["fold"])
    pair = (run_seed, fold)
    assert pair in EXPECTED_PAIRS and pair not in loaded_pairs
    checkpoint_path = ASSET_ROOT / item["checkpoint_path"]
    scaler_path = ASSET_ROOT / item["scaler_path"]
    assert sha256(checkpoint_path) == item["checkpoint_sha256"]
    assert sha256(scaler_path) == item["scaler_sha256"]
    scaler_asset = json.loads(scaler_path.read_text())
    assert scaler_asset["formal_source_sha"] == EXPECTED_FORMAL_SOURCE
    assert scaler_asset["seed"] == run_seed and scaler_asset["fold"] == fold
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert checkpoint["git_commit"] == EXPECTED_FORMAL_SOURCE
    assert checkpoint["benchmark_version"] == "nopp2025_train_v1"
    assert checkpoint["experiment_id"] == "stage5a_capacity_c128"
    assert checkpoint["model_config"] == {"hidden_dim": 128, "num_layers": 4, "dropout": 0.1}
    assert checkpoint["fold"] == fold and checkpoint["seed"] == run_seed + fold
    scaler = scaler_asset["normalizer"]
    assert checkpoint["normalizer"] == scaler
    assert set(scaler) == set(TARGETS)

    model = OwnGNNRepresentation(hidden_dim=128, num_layers=4, dropout=0.1)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.to(device)
    model.eval()
    fold_predictions = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            output = model(batch).detach().cpu().numpy().astype(np.float64)
            fold_predictions.extend(output.tolist())
    fold_predictions = np.asarray(fold_predictions, dtype=np.float64)
    assert fold_predictions.shape == (len(test), len(TARGETS))
    assert np.isfinite(fold_predictions).all()
    for j, target in enumerate(TARGETS):
        fold_predictions[:, j] = fold_predictions[:, j] * float(scaler[target]["std"]) + float(scaler[target]["mean"])
    assert np.isfinite(fold_predictions).all()
    accumulator += fold_predictions
    loaded_pairs.add(pair)
    loaded_hashes.append(item["checkpoint_sha256"])
    del checkpoint, model, fold_predictions
    torch.cuda.empty_cache()
    print(f"LOADED_MODEL seed={run_seed} fold={fold} count={len(loaded_pairs)}/25", flush=True)

assert loaded_pairs == EXPECTED_PAIRS and len(loaded_hashes) == 25
ensemble = accumulator / 25.0
assert ensemble.shape == (len(test), 5)
assert np.isfinite(ensemble).all()

submission = sample.copy(deep=True)
for j, target in enumerate(TARGETS):
    submission[target] = ensemble[:, j]
clean_submission = submission.copy(deep=True)
submission["Tg"] += DELTA
non_tg_targets = [target for target in TARGETS if target != "Tg"]
assert len(submission) == len(clean_submission) == len(sample) == len(test)
assert submission["id"].astype(str).tolist() == clean_submission["id"].astype(str).tolist() == sample["id"].astype(str).tolist()
assert submission.columns.tolist() == clean_submission.columns.tolist() == sample.columns.tolist()
np.testing.assert_array_equal(
    submission[non_tg_targets].to_numpy(dtype=np.float64),
    clean_submission[non_tg_targets].to_numpy(dtype=np.float64),
)
tg_delta = submission["Tg"].to_numpy(dtype=np.float64) - clean_submission["Tg"].to_numpy(dtype=np.float64)
assert tg_delta.shape == (len(test),) and np.isfinite(tg_delta).all()
np.testing.assert_allclose(tg_delta, np.full(len(test), DELTA, dtype=np.float64), rtol=0.0, atol=1e-10)
values = submission[TARGETS].to_numpy(dtype=np.float64)
assert np.isfinite(values).all() and not np.isnan(values).any() and not np.isinf(values).any()
submission_path = WORK / "submission.csv"
submission.to_csv(submission_path, index=False)
check = pd.read_csv(submission_path, dtype={"id": str})
assert len(check) == len(sample) == len(test)
assert check["id"].tolist() == sample["id"].astype(str).tolist()
assert check.columns.tolist() == sample.columns.tolist()
assert np.isfinite(check[TARGETS].to_numpy(dtype=np.float64)).all()
clean_non_tg_roundtrip = pd.read_csv(
    io.StringIO(clean_submission[non_tg_targets].to_csv(index=False))
)
np.testing.assert_array_equal(
    check[non_tg_targets].to_numpy(dtype=np.float64),
    clean_non_tg_roundtrip[non_tg_targets].to_numpy(dtype=np.float64),
)
check_tg_delta = check["Tg"].to_numpy(dtype=np.float64) - clean_submission["Tg"].to_numpy(dtype=np.float64)
np.testing.assert_allclose(check_tg_delta, np.full(len(test), DELTA, dtype=np.float64), rtol=0.0, atol=1e-10)
submission_sha = sha256(submission_path)
summary = {target: {"min": float(values[:, j].min()), "mean": float(values[:, j].mean()), "max": float(values[:, j].max())} for j, target in enumerate(TARGETS)}
report = {
    "classification": "POST-HOC / LEADERBOARD-PROBED / NOT VALID BLIND PERFORMANCE",
    "status": "passed",
    "tg_shift_celsius": DELTA,
    "clean_submission_comparison": {
        "rows_unchanged": len(submission) == len(clean_submission) == len(sample) == len(test),
        "id_order_unchanged": submission["id"].astype(str).tolist() == clean_submission["id"].astype(str).tolist() == sample["id"].astype(str).tolist(),
        "columns_unchanged": submission.columns.tolist() == clean_submission.columns.tolist() == sample.columns.tolist(),
        "non_tg_targets_exactly_equal": bool(np.array_equal(submission[non_tg_targets].to_numpy(dtype=np.float64), clean_submission[non_tg_targets].to_numpy(dtype=np.float64))),
        "csv_non_tg_exactly_equal_to_clean_serialization": True,
        "tg_delta_min": float(tg_delta.min()),
        "tg_delta_max": float(tg_delta.max()),
        "tg_delta_tolerance": 1e-10,
    },
    "formal_source_sha": EXPECTED_FORMAL_SOURCE,
    "main_sha": EXPECTED_MAIN,
    "model": "Stage 5A C128 Own-GNN; keep_dummy; GINEConv; 4 layers; mean|max pooling",
    "official_train_only": True,
    "external_or_supplementary_labels": False,
    "posthoc_clipping_or_calibration": False,
    "test_rows": len(test),
    "sample_submission_rows": len(sample),
    "id_order_match": True,
    "columns_match": True,
    "finite_predictions": True,
    "models_expected": 25,
    "models_loaded_and_ensembled": len(loaded_pairs),
    "loaded_pairs": [list(x) for x in sorted(loaded_pairs)],
    "loaded_checkpoint_sha256": loaded_hashes,
    "test_csv_sha256": sha256(test_path),
    "sample_submission_sha256": sha256(sample_path),
    "submission_sha256": submission_sha,
    "prediction_summary": summary,
    "runtime": {"python": sys.version.split()[0], "torch": torch.__version__, "torch_geometric": torch_geometric.__version__, "rdkit": rdkit.__version__, "accelerator": torch.cuda.get_device_name(device), "internet_enabled": False},
}
report_path = WORK / "clean_own_gnn_c128_run_report.json"
report_path.write_text(json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + "\n")
print("PREDICTION_RANGE", json.dumps(summary, sort_keys=True))
print(f"MODELS_LOADED_AND_ENSEMBLED={len(loaded_pairs)}/25")
print(f"SUBMISSION_CSV={submission_path} SHA256={submission_sha}")
print(f"EXECUTION_REPORT={report_path}")
print(f"POSTHOC_DIAGNOSTIC_TG_SHIFT_C={DELTA:.0f}")
print(f"ASSERTIONS_PASSED: ids, columns, row_count, finite_targets, 25/25 models, non-Tg exact, Tg delta ±1e-10")
